In [2]:
import tkinter as tk

root = tk.Tk()

# Setting some window properties
root.title("Tk Example")
root.configure(background="yellow")
root.minsize(200, 200)
root.maxsize(500, 500)
root.geometry("300x300+50+50")

root.mainloop()

In [ ]:
# https://stackoverflow.com/questions/41656176/tkinter-canvas-zoom-move-pan

In [3]:
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageTk

class MatrixTextEditor:
    def __init__(self, root, file_path):
        self.root = root
        self.root.title("Matrix Text File Editor (Zoom & Pan)")
        self.file_path = file_path
        
        # 1. Load the matrix from the text file using NumPy
        try:
            self.matrix = np.loadtxt(self.file_path, dtype=int)
        except Exception as e:
            print(f"Error loading matrix: {e}")
            self.matrix = np.zeros((100, 100), dtype=int) # Fallback blank matrix

        self.height, self.width = self.matrix.shape
        
        # Normalize matrix values for display if they are 0 and 1
        self.val_max = self.matrix.max()
        if self.val_max <= 1:
            self.display_scale = 255
        else:
            self.display_scale = 1
            
        self.brush_size = 3
        self.draw_value = 0  # Default to drawing black (erasing artifacts)
        
        # Zoom & Pan State variables
        self.zoom_scale = 1.0
        self.pan_x = 0
        self.pan_y = 0
        self.drag_start_x = 0
        self.drag_start_y = 0
        
        self.setup_ui()
        self.update_canvas()

    def setup_ui(self):
        # Control Panel
        control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
        control_frame.pack(side=tk.TOP, fill=tk.X)
        
        tk.Button(control_frame, text="Eraser / Black (0)", command=lambda: self.set_draw_value(0), bg="#ffcccc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Button(control_frame, text="Draw / White (1/Max)", command=lambda: self.set_draw_value(1), bg="#ccffcc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        
        tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(side=tk.LEFT, padx=10)
        self.size_slider = tk.Scale(control_frame, from_=1, to=20, orient=tk.HORIZONTAL, command=self.update_brush)
        self.size_slider.set(self.brush_size)
        self.size_slider.pack(side=tk.LEFT, padx=5)
        
        tk.Button(control_frame, text="Reset View", command=self.reset_view, bg="#e0e0e0", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=10)
        tk.Button(control_frame, text="Save Matrix to .txt", command=self.save_matrix, bg="#cce5ff", font=("Arial", 10, "bold")).pack(side=tk.RIGHT, padx=5)

        # Scrollable / Canvas Area for the Image
        canvas_frame = tk.Frame(self.root)
        canvas_frame.pack(fill=tk.BOTH, expand=True)
        
        self.canvas = tk.Canvas(canvas_frame, bg="gray")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Bind Mouse Actions for Painting (Left Click)
        self.canvas.bind("<Button-1>", self.paint)      # Click
        self.canvas.bind("<B1-Motion>", self.paint)    # Click and drag

        # Bind Mouse Actions for Panning (Right Click)
        self.canvas.bind("<Button-3>", self.start_pan)
        self.canvas.bind("<B3-Motion>", self.do_pan)

        # Bind Mouse Wheel for Zooming
        self.canvas.bind("<MouseWheel>", self.zoom)     # Windows and macOS
        self.canvas.bind("<Button-4>", self.zoom)       # Linux scroll up
        self.canvas.bind("<Button-5>", self.zoom)       # Linux scroll down

    def set_draw_value(self, val):
        if val == 1 and self.val_max > 1:
            self.draw_value = self.val_max
        else:
            self.draw_value = val

    def update_brush(self, val):
        self.brush_size = int(val)

    def reset_view(self):
        self.zoom_scale = 1.0
        self.pan_x = 0
        self.pan_y = 0
        self.update_canvas()

    def update_canvas(self):
        # Scale matrix for display
        scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
        self.pil_img = Image.fromarray(scaled_matrix)
        
        # Resize using NEAREST interpolation to keep pixels sharp during zoom
        new_width = max(1, int(self.width * self.zoom_scale))
        new_height = max(1, int(self.height * self.zoom_scale))
        pil_img_resized = self.pil_img.resize((new_width, new_height), Image.NEAREST)
        
        self.tk_img = ImageTk.PhotoImage(pil_img_resized)
        
        # Redraw on canvas
        self.canvas.delete("all")
        self.canvas.create_image(self.pan_x, self.pan_y, anchor=tk.NW, image=self.tk_img)

    def start_pan(self, event):
        self.drag_start_x = event.x
        self.drag_start_y = event.y

    def do_pan(self, event):
        dx = event.x - self.drag_start_x
        dy = event.y - self.drag_start_y
        self.pan_x += dx
        self.pan_y += dy
        self.drag_start_x = event.x
        self.drag_start_y = event.y
        self.update_canvas()

    def zoom(self, event):
        # Determine zoom direction
        if event.num == 4 or event.delta > 0:
            zoom_factor = 1.15
        elif event.num == 5 or event.delta < 0:
            zoom_factor = 1 / 1.15
        else:
            zoom_factor = 1.0

        old_zoom = self.zoom_scale
        self.zoom_scale *= zoom_factor
        
        # Clamp zoom limits
        if self.zoom_scale < 0.1:
            self.zoom_scale = 0.1
        elif self.zoom_scale > 50.0:
            self.zoom_scale = 50.0

        # Adjust pan so zoom centers around mouse pointer
        self.pan_x = event.x - (event.x - self.pan_x) * (self.zoom_scale / old_zoom)
        self.pan_y = event.y - (event.y - self.pan_y) * (self.zoom_scale / old_zoom)

        self.update_canvas()

    def paint(self, event):
        # Map canvas window coordinates to matrix/image coordinates
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        
        x, y = int(img_x), int(img_y)
        r = max(1, int(self.brush_size / self.zoom_scale))
        
        # Update matrix slice around coordinates
        y1, y2 = max(0, y - r), min(self.height, y + r)
        x1, x2 = max(0, x - r), min(self.width, x + r)
        
        matrix_val = self.draw_value if self.display_scale == 1 else (1 if self.draw_value > 0 else 0)
        self.matrix[y1:y2, x1:x2] = matrix_val
        
        self.update_canvas()

    def save_matrix(self):
        save_path = filedialog.asksaveasfilename(
            defaultextension=".txt",
            initialfile=self.file_path,
            filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
        )
        if save_path:
            np.savetxt(save_path, self.matrix, fmt="%d")
            print(f"Matrix successfully saved to {save_path}")

if __name__ == "__main__":
    root = tk.Tk()
    root.geometry("900x700")
    
    target_txt_file = r"../file_io/line_img_matrix.txt"
    
    app = MatrixTextEditor(root, target_txt_file)
    root.mainloop()

In [6]:
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageTk

class MatrixTextEditor:
    def __init__(self, root, file_path):
        self.root = root
        self.root.title("Matrix Text File Editor (Fixed Zoom & Pan)")
        self.file_path = file_path
        
        # 1. Load the matrix from the text file using NumPy
        try:
            self.matrix = np.loadtxt(self.file_path, dtype=int)
        except Exception as e:
            print(f"Error loading matrix: {e}")
            self.matrix = np.zeros((100, 100), dtype=int)

        self.height, self.width = self.matrix.shape
        
        # Normalize matrix values for display if they are 0 and 1
        self.val_max = self.matrix.max()
        if self.val_max <= 1:
            self.display_scale = 255
        else:
            self.display_scale = 1
            
        self.brush_size = 3
        self.draw_value = 0  # Default to drawing black (erasing artifacts)
        
        # Zoom & Pan State variables
        self.zoom_scale = 1.0
        self.pan_x = 0.0
        self.pan_y = 0.0
        self.drag_start_x = 0
        self.drag_start_y = 0
        
        # Caching for performance
        self._cached_zoom = None
        self._cached_tk_img = None
        
        self.setup_ui()
        self.update_canvas()

    def setup_ui(self):
        # Control Panel
        control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
        control_frame.pack(side=tk.TOP, fill=tk.X)
        
        tk.Button(control_frame, text="Eraser / Black (0)", command=lambda: self.set_draw_value(0), bg="#ffcccc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Button(control_frame, text="Draw / White (1/Max)", command=lambda: self.set_draw_value(1), bg="#ccffcc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        
        tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(side=tk.LEFT, padx=10)
        self.size_slider = tk.Scale(control_frame, from_=1, to=20, orient=tk.HORIZONTAL, command=self.update_brush)
        self.size_slider.set(self.brush_size)
        self.size_slider.pack(side=tk.LEFT, padx=5)
        
        tk.Button(control_frame, text="Reset View", command=self.reset_view, bg="#e0e0e0", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=10)
        tk.Button(control_frame, text="Save Matrix to .txt", command=self.save_matrix, bg="#cce5ff", font=("Arial", 10, "bold")).pack(side=tk.RIGHT, padx=5)

        # Canvas Area for the Image
        canvas_frame = tk.Frame(self.root)
        canvas_frame.pack(fill=tk.BOTH, expand=True)
        
        self.canvas = tk.Canvas(canvas_frame, bg="gray", highlightthickness=0)
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Bind Mouse Actions for Painting (Left Click)
        self.canvas.bind("<Button-1>", self.paint)      
        self.canvas.bind("<B1-Motion>", self.paint)    

        # Bind Mouse Actions for Panning (Scrollwheel / Middle Click)
        self.canvas.bind("<Button-2>", self.start_pan)
        self.canvas.bind("<B2-Motion>", self.do_pan)
        self.canvas.bind("<Button-3>", self.start_pan)
        self.canvas.bind("<B3-Motion>", self.do_pan)

        # Bind Mouse Wheel for Zooming
        self.canvas.bind("<MouseWheel>", self.zoom)     # Windows and macOS
        self.canvas.bind("<Button-4>", self.zoom)       # Linux scroll up
        self.canvas.bind("<Button-5>", self.zoom)       # Linux scroll down

    def set_draw_value(self, val):
        if val == 1 and self.val_max > 1:
            self.draw_value = self.val_max
        else:
            self.draw_value = val

    def update_brush(self, val):
        self.brush_size = int(val)

    def reset_view(self):
        self.zoom_scale = 1.0
        self.pan_x = 0.0
        self.pan_y = 0.0
        self.update_canvas()

    def update_canvas(self, matrix_changed=False):
        scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
        pil_img = Image.fromarray(scaled_matrix)
        
        new_width = max(1, int(self.width * self.zoom_scale))
        new_height = max(1, int(self.height * self.zoom_scale))
        
        # Regenerate image if zoom level changed, matrix updated, or cache is empty
        if matrix_changed or self._cached_zoom != self.zoom_scale or self._cached_tk_img is None:
            resized_pil = pil_img.resize((new_width, new_height), Image.NEAREST)
            self._cached_tk_img = ImageTk.PhotoImage(resized_pil)
            self._cached_zoom = self.zoom_scale

        self.canvas.delete("all")
        self.image_item = self.canvas.create_image(self.pan_x, self.pan_y, anchor=tk.NW, image=self._cached_tk_img)

    def start_pan(self, event):
        self.drag_start_x = event.x
        self.drag_start_y = event.y

    def do_pan(self, event):
        dx = event.x - self.drag_start_x
        dy = event.y - self.drag_start_y
        self.pan_x += dx
        self.pan_y += dy
        self.drag_start_x = event.x
        self.drag_start_y = event.y
        
        self.canvas.coords(self.image_item, self.pan_x, self.pan_y)

    def zoom(self, event):
        if event.num == 4 or event.delta > 0:
            zoom_factor = 1.15
        elif event.num == 5 or event.delta < 0:
            zoom_factor = 1 / 1.15
        else:
            zoom_factor = 1.0

        old_zoom = self.zoom_scale
        new_zoom = self.zoom_scale * zoom_factor
        
        # Clamp zoom limits
        if new_zoom < 0.1 or new_zoom > 50.0:
            return

        self.zoom_scale = new_zoom

        # Adjust pan so zoom precisely centers around mouse pointer coordinates
        self.pan_x = event.x - (event.x - self.pan_x) * (self.zoom_scale / old_zoom)
        self.pan_y = event.y - (event.y - self.pan_y) * (self.zoom_scale / old_zoom)

        self.update_canvas()

    def paint(self, event):
        # Map canvas window coordinates precisely to matrix indices
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        
        x, y = int(img_x), int(img_y)
        r = self.brush_size
        
        y1, y2 = max(0, y - r), min(self.height, y + r)
        x1, x2 = max(0, x - r), min(self.width, x + r)
        
        matrix_val = self.draw_value if self.display_scale == 1 else (1 if self.draw_value > 0 else 0)
        self.matrix[y1:y2, x1:x2] = matrix_val
        
        # Update canvas with matrix change flagged
        self.update_canvas(matrix_changed=True)

    def save_matrix(self):
        save_path = filedialog.asksaveasfilename(
            defaultextension=".txt",
            initialfile=self.file_path,
            filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
        )
        if save_path:
            np.savetxt(save_path, self.matrix, fmt="%d")
            print(f"Matrix successfully saved to {save_path}")

if __name__ == "__main__":
    root = tk.Tk()
    root.geometry("900x700")
    
    target_txt_file = r"../file_io/line_img_matrix.txt"
    
    app = MatrixTextEditor(root, target_txt_file)
    root.mainloop()

In [7]:
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageTk

class MatrixTextEditor:
    def __init__(self, root, file_path):
        self.root = root
        self.root.title("Matrix Text File Editor (Continuous Strokes)")
        self.file_path = file_path
        
        # 1. Load the matrix from the text file using NumPy
        try:
            self.matrix = np.loadtxt(self.file_path, dtype=int)
        except Exception as e:
            print(f"Error loading matrix: {e}")
            self.matrix = np.zeros((100, 100), dtype=int)

        self.height, self.width = self.matrix.shape
        
        # Normalize matrix values for display if they are 0 and 1
        self.val_max = self.matrix.max()
        if self.val_max <= 1:
            self.display_scale = 255
        else:
            self.display_scale = 1
            
        self.brush_size = 3
        self.draw_value = 0  # Default to drawing black (erasing artifacts)
        
        # Zoom & Pan State variables
        self.zoom_scale = 1.0
        self.pan_x = 0.0
        self.pan_y = 0.0
        self.drag_start_x = 0
        self.drag_start_y = 0
        
        # Stroke tracking for interpolation
        self.last_img_x = None
        self.last_img_y = None
        
        # Caching for performance
        self._cached_zoom = None
        self._cached_tk_img = None
        
        self.setup_ui()
        self.update_canvas()

    def setup_ui(self):
        # Control Panel
        control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
        control_frame.pack(side=tk.TOP, fill=tk.X)
        
        tk.Button(control_frame, text="Eraser / Black (0)", command=lambda: self.set_draw_value(0), bg="#ffcccc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Button(control_frame, text="Draw / White (1/Max)", command=lambda: self.set_draw_value(1), bg="#ccffcc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        
        tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(side=tk.LEFT, padx=10)
        self.size_slider = tk.Scale(control_frame, from_=1, to=20, orient=tk.HORIZONTAL, command=self.update_brush)
        self.size_slider.set(self.brush_size)
        self.size_slider.pack(side=tk.LEFT, padx=5)
        
        tk.Button(control_frame, text="Reset View", command=self.reset_view, bg="#e0e0e0", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=10)
        tk.Button(control_frame, text="Save Matrix to .txt", command=self.save_matrix, bg="#cce5ff", font=("Arial", 10, "bold")).pack(side=tk.RIGHT, padx=5)

        # Canvas Area for the Image
        canvas_frame = tk.Frame(self.root)
        canvas_frame.pack(fill=tk.BOTH, expand=True)
        
        self.canvas = tk.Canvas(canvas_frame, bg="gray", highlightthickness=0)
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Bind Mouse Actions for Painting (Left Click & Drag)
        self.canvas.bind("<Button-1>", self.start_stroke)      
        self.canvas.bind("<B1-Motion>", self.paint)    

        # Bind Mouse Actions for Panning (Scrollwheel / Middle Click / Right Click)
        self.canvas.bind("<Button-2>", self.start_pan)
        self.canvas.bind("<B2-Motion>", self.do_pan)
        self.canvas.bind("<Button-3>", self.start_pan)
        self.canvas.bind("<B3-Motion>", self.do_pan)

        # Bind Mouse Wheel for Zooming
        self.canvas.bind("<MouseWheel>", self.zoom)     # Windows and macOS
        self.canvas.bind("<Button-4>", self.zoom)       # Linux scroll up
        self.canvas.bind("<Button-5>", self.zoom)       # Linux scroll down

    def set_draw_value(self, val):
        if val == 1 and self.val_max > 1:
            self.draw_value = self.val_max
        else:
            self.draw_value = val

    def update_brush(self, val):
        self.brush_size = int(val)

    def reset_view(self):
        self.zoom_scale = 1.0
        self.pan_x = 0.0
        self.pan_y = 0.0
        self.update_canvas()

    def update_canvas(self, matrix_changed=False):
        scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
        pil_img = Image.fromarray(scaled_matrix)
        
        new_width = max(1, int(self.width * self.zoom_scale))
        new_height = max(1, int(self.height * self.zoom_scale))
        
        if matrix_changed or self._cached_zoom != self.zoom_scale or self._cached_tk_img is None:
            resized_pil = pil_img.resize((new_width, new_height), Image.NEAREST)
            self._cached_tk_img = ImageTk.PhotoImage(resized_pil)
            self._cached_zoom = self.zoom_scale

        self.canvas.delete("all")
        self.image_item = self.canvas.create_image(self.pan_x, self.pan_y, anchor=tk.NW, image=self._cached_tk_img)

    def start_pan(self, event):
        self.drag_start_x = event.x
        self.drag_start_y = event.y

    def do_pan(self, event):
        dx = event.x - self.drag_start_x
        dy = event.y - self.drag_start_y
        self.pan_x += dx
        self.pan_y += dy
        self.drag_start_x = event.x
        self.drag_start_y = event.y
        
        self.canvas.coords(self.image_item, self.pan_x, self.pan_y)

    def zoom(self, event):
        if event.num == 4 or event.delta > 0:
            zoom_factor = 1.15
        elif event.num == 5 or event.delta < 0:
            zoom_factor = 1 / 1.15
        else:
            zoom_factor = 1.0

        old_zoom = self.zoom_scale
        new_zoom = self.zoom_scale * zoom_factor
        
        if new_zoom < 0.1 or new_zoom > 50.0:
            return

        self.zoom_scale = new_zoom
        self.pan_x = event.x - (event.x - self.pan_x) * (self.zoom_scale / old_zoom)
        self.pan_y = event.y - (event.y - self.pan_y) * (self.zoom_scale / old_zoom)

        self.update_canvas()

    def start_stroke(self, event):
        # Initialize starting coordinate for a brand new stroke
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        self.last_img_x = img_x
        self.last_img_y = img_y
        self.paint(event)

    def paint(self, event):
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        
        if self.last_img_x is None or self.last_img_y is None:
            self.last_img_x = img_x
            self.last_img_y = img_y

        # Calculate number of interpolation steps based on distance
        dist = np.hypot(img_x - self.last_img_x, img_y - self.last_img_y)
        steps = max(1, int(dist))
        
        xs = np.linspace(self.last_img_x, img_x, steps)
        ys = np.linspace(self.last_img_y, img_y, steps)
        
        matrix_val = self.draw_value if self.display_scale == 1 else (1 if self.draw_value > 0 else 0)
        r = self.brush_size
        
        # Paint across all interpolated points between last position and current position
        for cx, cy in zip(xs, ys):
            x, y = int(cx), int(cy)
            y1, y2 = max(0, y - r), min(self.height, y + r)
            x1, x2 = max(0, x - r), min(self.width, x + r)
            self.matrix[y1:y2, x1:x2] = matrix_val
            
        self.last_img_x = img_x
        self.last_img_y = img_y
        
        self.update_canvas(matrix_changed=True)

    def save_matrix(self):
        save_path = filedialog.asksaveasfilename(
            defaultextension=".txt",
            initialfile=self.file_path,
            filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
        )
        if save_path:
            np.savetxt(save_path, self.matrix, fmt="%d")
            print(f"Matrix successfully saved to {save_path}")

if __name__ == "__main__":
    root = tk.Tk()
    root.geometry("900x700")
    
    target_txt_file = r"../file_io/line_img_matrix.txt"
    
    app = MatrixTextEditor(root, target_txt_file)
    root.mainloop()

In [1]:
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageTk

class MatrixTextEditor:
    def __init__(self, root, file_path):
        self.root = root
        self.root.title("Matrix Text File Editor (Live Brush Preview)")
        self.file_path = file_path
        
        # 1. Load the matrix from the text file using NumPy
        try:
            self.matrix = np.loadtxt(self.file_path, dtype=int)
        except Exception as e:
            print(f"Error loading matrix: {e}")
            self.matrix = np.zeros((100, 100), dtype=int)

        self.height, self.width = self.matrix.shape
        
        # Normalize matrix values for display if they are 0 and 1
        self.val_max = self.matrix.max()
        if self.val_max <= 1:
            self.display_scale = 255
        else:
            self.display_scale = 1
            
        self.brush_size = 3
        self.draw_value = 0  # Default to drawing black (erasing artifacts)
        
        # Zoom & Pan State variables
        self.zoom_scale = 1.0
        self.pan_x = 0.0
        self.pan_y = 0.0
        self.drag_start_x = 0
        self.drag_start_y = 0
        
        # Stroke tracking for interpolation
        self.last_img_x = None
        self.last_img_y = None
        
        # Cursor preview item reference
        self.cursor_rect = None
        
        # Caching for performance
        self._cached_zoom = None
        self._cached_tk_img = None
        
        self.setup_ui()
        self.update_canvas()

    def setup_ui(self):
        # Control Panel
        control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
        control_frame.pack(side=tk.TOP, fill=tk.X)
        
        tk.Button(control_frame, text="Eraser / Black (0)", command=lambda: self.set_draw_value(0), bg="#ffcccc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Button(control_frame, text="Draw / White (1/Max)", command=lambda: self.set_draw_value(1), bg="#ccffcc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        
        tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(side=tk.LEFT, padx=10)
        self.size_slider = tk.Scale(control_frame, from_=1, to=20, orient=tk.HORIZONTAL, command=self.update_brush)
        self.size_slider.set(self.brush_size)
        self.size_slider.pack(side=tk.LEFT, padx=5)
        
        tk.Button(control_frame, text="Reset View", command=self.reset_view, bg="#e0e0e0", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=10)
        tk.Button(control_frame, text="Save Matrix to .txt", command=self.save_matrix, bg="#cce5ff", font=("Arial", 10, "bold")).pack(side=tk.RIGHT, padx=5)

        # Canvas Area for the Image
        canvas_frame = tk.Frame(self.root)
        canvas_frame.pack(fill=tk.BOTH, expand=True)
        
        self.canvas = tk.Canvas(canvas_frame, bg="gray", highlightthickness=0)
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Bind Mouse Actions for Painting (Left Click & Drag)
        self.canvas.bind("<Button-1>", self.start_stroke)      
        self.canvas.bind("<B1-Motion>", self.paint)    

        # Bind Mouse Actions for Panning (Scrollwheel / Middle Click / Right Click)
        self.canvas.bind("<Button-2>", self.start_pan)
        self.canvas.bind("<B2-Motion>", self.do_pan)
        self.canvas.bind("<Button-3>", self.start_pan)
        self.canvas.bind("<B3-Motion>", self.do_pan)

        # Bind Mouse Wheel for Zooming
        self.canvas.bind("<MouseWheel>", self.zoom)     # Windows and macOS
        self.canvas.bind("<Button-4>", self.zoom)       # Linux scroll up
        self.canvas.bind("<Button-5>", self.zoom)       # Linux scroll down

        # Bind Cursor Hover Preview Events
        self.canvas.bind("<Motion>", self.update_cursor)
        self.canvas.bind("<Enter>", self.show_cursor)
        self.canvas.bind("<Leave>", self.hide_cursor)

    def set_draw_value(self, val):
        if val == 1 and self.val_max > 1:
            self.draw_value = self.val_max
        else:
            self.draw_value = val

    def update_brush(self, val):
        self.brush_size = int(val)

    def reset_view(self):
        self.zoom_scale = 1.0
        self.pan_x = 0.0
        self.pan_y = 0.0
        self.update_canvas()

    def update_canvas(self, matrix_changed=False):
        scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
        pil_img = Image.fromarray(scaled_matrix)
        
        new_width = max(1, int(self.width * self.zoom_scale))
        new_height = max(1, int(self.height * self.zoom_scale))
        
        if matrix_changed or self._cached_zoom != self.zoom_scale or self._cached_tk_img is None:
            resized_pil = pil_img.resize((new_width, new_height), Image.NEAREST)
            self._cached_tk_img = ImageTk.PhotoImage(resized_pil)
            self._cached_zoom = self.zoom_scale

        self.canvas.delete("all")
        self.image_item = self.canvas.create_image(self.pan_x, self.pan_y, anchor=tk.NW, image=self._cached_tk_img)
        
        # Recreate cursor preview rect if active
        if self.cursor_rect is not None:
            self.cursor_rect = self.canvas.create_rectangle(0, 0, 0, 0, outline="red", width=1)

    def show_cursor(self, event):
        if self.cursor_rect is None:
            self.cursor_rect = self.canvas.create_rectangle(0, 0, 0, 0, outline="red", width=1)
        self.update_cursor(event)

    def hide_cursor(self, event):
        if self.cursor_rect is not None:
            self.canvas.delete(self.cursor_rect)
            self.cursor_rect = None

    def update_cursor(self, event):
        if self.cursor_rect is None:
            self.show_cursor(event)
            
        # Map canvas coordinates to image coordinates
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        
        x, y = int(img_x), int(img_y)
        r = self.brush_size
        
        # Calculate canvas bounding box for the brush square footprint
        x1 = (x - r) * self.zoom_scale + self.pan_x
        y1 = (y - r) * self.zoom_scale + self.pan_y
        x2 = (x + r) * self.zoom_scale + self.pan_x
        y2 = (y + r) * self.zoom_scale + self.pan_y
        
        self.canvas.coords(self.cursor_rect, x1, y1, x2, y2)

    def start_pan(self, event):
        self.drag_start_x = event.x
        self.drag_start_y = event.y

    def do_pan(self, event):
        dx = event.x - self.drag_start_x
        dy = event.y - self.drag_start_y
        self.pan_x += dx
        self.pan_y += dy
        self.drag_start_x = event.x
        self.drag_start_y = event.y
        
        self.canvas.coords(self.image_item, self.pan_x, self.pan_y)
        self.update_cursor(event)

    def zoom(self, event):
        if event.num == 4 or event.delta > 0:
            zoom_factor = 1.15
        elif event.num == 5 or event.delta < 0:
            zoom_factor = 1 / 1.15
        else:
            zoom_factor = 1.0

        old_zoom = self.zoom_scale
        new_zoom = self.zoom_scale * zoom_factor
        
        if new_zoom < 0.1 or new_zoom > 50.0:
            return

        self.zoom_scale = new_zoom
        self.pan_x = event.x - (event.x - self.pan_x) * (self.zoom_scale / old_zoom)
        self.pan_y = event.y - (event.y - self.pan_y) * (self.zoom_scale / old_zoom)

        self.update_canvas(matrix_changed=False)
        self.update_cursor(event)

    def start_stroke(self, event):
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        self.last_img_x = img_x
        self.last_img_y = img_y
        self.paint(event)

    def paint(self, event):
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        
        if self.last_img_x is None or self.last_img_y is None:
            self.last_img_x = img_x
            self.last_img_y = img_y

        dist = np.hypot(img_x - self.last_img_x, img_y - self.last_img_y)
        steps = max(1, int(dist))
        
        xs = np.linspace(self.last_img_x, img_x, steps)
        ys = np.linspace(self.last_img_y, img_y, steps)
        
        matrix_val = self.draw_value if self.display_scale == 1 else (1 if self.draw_value > 0 else 0)
        r = self.brush_size
        
        for cx, cy in zip(xs, ys):
            x, y = int(cx), int(cy)
            y1, y2 = max(0, y - r), min(self.height, y + r)
            x1, x2 = max(0, x - r), min(self.width, x + r)
            self.matrix[y1:y2, x1:x2] = matrix_val
            
        self.last_img_x = img_x
        self.last_img_y = img_y
        
        self.update_canvas(matrix_changed=True)
        self.update_cursor(event)

    def save_matrix(self):
        save_path = filedialog.asksaveasfilename(
            defaultextension=".txt",
            initialfile=self.file_path,
            filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
        )
        if save_path:
            np.savetxt(save_path, self.matrix, fmt="%d")
            print(f"Matrix successfully saved to {save_path}")

if __name__ == "__main__":
    root = tk.Tk()
    root.geometry("900x700")
    
    target_txt_file = r"../file_io/line_img_matrix.txt"
    
    app = MatrixTextEditor(root, target_txt_file)
    root.mainloop()

In [2]:
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageTk

class MatrixTextEditor:
    def __init__(self, root, file_path):
        self.root = root
        self.root.title("Matrix Text File Editor (Grid Lines & Hidden Cursor)")
        self.file_path = file_path
        
        # 1. Load the matrix from the text file using NumPy
        try:
            self.matrix = np.loadtxt(self.file_path, dtype=int)
        except Exception as e:
            print(f"Error loading matrix: {e}")
            self.matrix = np.zeros((100, 100), dtype=int)

        self.height, self.width = self.matrix.shape
        
        # Normalize matrix values for display if they are 0 and 1
        self.val_max = self.matrix.max()
        if self.val_max <= 1:
            self.display_scale = 255
        else:
            self.display_scale = 1
            
        self.brush_size = 3
        self.draw_value = 0  # Default to drawing black (erasing artifacts)
        
        # Zoom & Pan State variables
        self.zoom_scale = 1.0
        self.pan_x = 0.0
        self.pan_y = 0.0
        self.drag_start_x = 0
        self.drag_start_y = 0
        
        # Stroke tracking for interpolation
        self.last_img_x = None
        self.last_img_y = None
        
        # Cursor preview item reference
        self.cursor_rect = None
        
        # Caching for performance
        self._cached_zoom = None
        self._cached_tk_img = None
        
        self.setup_ui()
        self.update_canvas()

    def setup_ui(self):
        # Control Panel
        control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
        control_frame.pack(side=tk.TOP, fill=tk.X)
        
        tk.Button(control_frame, text="Eraser / Black (0)", command=lambda: self.set_draw_value(0), bg="#ffcccc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Button(control_frame, text="Draw / White (1/Max)", command=lambda: self.set_draw_value(1), bg="#ccffcc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        
        tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(side=tk.LEFT, padx=10)
        self.size_slider = tk.Scale(control_frame, from_=1, to=20, orient=tk.HORIZONTAL, command=self.update_brush)
        self.size_slider.set(self.brush_size)
        self.size_slider.pack(side=tk.LEFT, padx=5)
        
        tk.Button(control_frame, text="Reset View", command=self.reset_view, bg="#e0e0e0", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=10)
        tk.Button(control_frame, text="Save Matrix to .txt", command=self.save_matrix, bg="#cce5ff", font=("Arial", 10, "bold")).pack(side=tk.RIGHT, padx=5)

        # Canvas Area for the Image
        canvas_frame = tk.Frame(self.root)
        canvas_frame.pack(fill=tk.BOTH, expand=True)
        
        self.canvas = tk.Canvas(canvas_frame, bg="gray", highlightthickness=0)
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Bind Mouse Actions for Painting (Left Click & Drag)
        self.canvas.bind("<Button-1>", self.start_stroke)      
        self.canvas.bind("<B1-Motion>", self.paint)    

        # Bind Mouse Actions for Panning (Scrollwheel / Middle Click / Right Click)
        self.canvas.bind("<Button-2>", self.start_pan)
        self.canvas.bind("<B2-Motion>", self.do_pan)
        self.canvas.bind("<Button-3>", self.start_pan)
        self.canvas.bind("<B3-Motion>", self.do_pan)

        # Bind Mouse Wheel for Zooming
        self.canvas.bind("<MouseWheel>", self.zoom)     # Windows and macOS
        self.canvas.bind("<Button-4>", self.zoom)       # Linux scroll up
        self.canvas.bind("<Button-5>", self.zoom)       # Linux scroll down

        # Bind Cursor Hover Preview Events
        self.canvas.bind("<Motion>", self.update_cursor)
        self.canvas.bind("<Enter>", self.show_cursor)
        self.canvas.bind("<Leave>", self.hide_cursor)

    def set_draw_value(self, val):
        if val == 1 and self.val_max > 1:
            self.draw_value = self.val_max
        else:
            self.draw_value = val

    def update_brush(self, val):
        self.brush_size = int(val)

    def reset_view(self):
        self.zoom_scale = 1.0
        self.pan_x = 0.0
        self.pan_y = 0.0
        self.update_canvas()

    def update_canvas(self, matrix_changed=False):
        scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
        pil_img = Image.fromarray(scaled_matrix)
        
        new_width = max(1, int(self.width * self.zoom_scale))
        new_height = max(1, int(self.height * self.zoom_scale))
        
        if matrix_changed or self._cached_zoom != self.zoom_scale or self._cached_tk_img is None:
            resized_pil = pil_img.resize((new_width, new_height), Image.NEAREST)
            self._cached_tk_img = ImageTk.PhotoImage(resized_pil)
            self._cached_zoom = self.zoom_scale

        self.canvas.delete("all")
        self.image_item = self.canvas.create_image(self.pan_x, self.pan_y, anchor=tk.NW, image=self._cached_tk_img)
        
        # Automatically draw grid lines if zoomed in enough (cells >= 8 pixels wide)
        if self.zoom_scale >= 8.0:
            grid_color = "#333333" if self.display_scale == 255 else "#cccccc"
            
            # Vertical grid lines
            for x in range(self.width + 1):
                gx = x * self.zoom_scale + self.pan_x
                self.canvas.create_line(gx, self.pan_y, gx, self.pan_y + new_height, fill=grid_color)
                
            # Horizontal grid lines
            for y in range(self.height + 1):
                gy = y * self.zoom_scale + self.pan_y
                self.canvas.create_line(self.pan_x, gy, self.pan_x + new_width, gy, fill=grid_color)

        # Recreate cursor preview rect if active
        if self.cursor_rect is not None:
            self.cursor_rect = self.canvas.create_rectangle(0, 0, 0, 0, outline="red", width=1)

    def show_cursor(self, event):
        # Hide standard mouse cursor by setting it to a transparent or blank string on Windows/Linux/macOS
        self.canvas.config(cursor="none")
        if self.cursor_rect is None:
            self.cursor_rect = self.canvas.create_rectangle(0, 0, 0, 0, outline="red", width=1)
        self.update_cursor(event)

    def hide_cursor(self, event):
        # Restore standard mouse cursor when leaving canvas area
        self.canvas.config(cursor="")
        if self.cursor_rect is not None:
            self.canvas.delete(self.cursor_rect)
            self.cursor_rect = None

    def update_cursor(self, event):
        if self.cursor_rect is None:
            self.show_cursor(event)
            
        # Map canvas coordinates to image coordinates
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        
        x, y = int(img_x), int(img_y)
        r = self.brush_size
        
        # Calculate canvas bounding box for the brush square footprint
        x1 = (x - r) * self.zoom_scale + self.pan_x
        y1 = (y - r) * self.zoom_scale + self.pan_y
        x2 = (x + r) * self.zoom_scale + self.pan_x
        y2 = (y + r) * self.zoom_scale + self.pan_y
        
        self.canvas.coords(self.cursor_rect, x1, y1, x2, y2)

    def start_pan(self, event):
        self.drag_start_x = event.x
        self.drag_start_y = event.y

    def do_pan(self, event):
        dx = event.x - self.drag_start_x
        dy = event.y - self.drag_start_y
        self.pan_x += dx
        self.pan_y += dy
        self.drag_start_x = event.x
        self.drag_start_y = event.y
        
        self.update_canvas(matrix_changed=False)
        self.update_cursor(event)

    def zoom(self, event):
        if event.num == 4 or event.delta > 0:
            zoom_factor = 1.15
        elif event.num == 5 or event.delta < 0:
            zoom_factor = 1 / 1.15
        else:
            zoom_factor = 1.0

        old_zoom = self.zoom_scale
        new_zoom = self.zoom_scale * zoom_factor
        
        if new_zoom < 0.1 or new_zoom > 50.0:
            return

        self.zoom_scale = new_zoom
        self.pan_x = event.x - (event.x - self.pan_x) * (self.zoom_scale / old_zoom)
        self.pan_y = event.y - (event.y - self.pan_y) * (self.zoom_scale / old_zoom)

        self.update_canvas(matrix_changed=False)
        self.update_cursor(event)

    def start_stroke(self, event):
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        self.last_img_x = img_x
        self.last_img_y = img_y
        self.paint(event)

    def paint(self, event):
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        
        if self.last_img_x is None or self.last_img_y is None:
            self.last_img_x = img_x
            self.last_img_y = img_y

        dist = np.hypot(img_x - self.last_img_x, img_y - self.last_img_y)
        steps = max(1, int(dist))
        
        xs = np.linspace(self.last_img_x, img_x, steps)
        ys = np.linspace(self.last_img_y, img_y, steps)
        
        matrix_val = self.draw_value if self.display_scale == 1 else (1 if self.draw_value > 0 else 0)
        r = self.brush_size
        
        for cx, cy in zip(xs, ys):
            x, y = int(cx), int(cy)
            y1, y2 = max(0, y - r), min(self.height, y + r)
            x1, x2 = max(0, x - r), min(self.width, x + r)
            self.matrix[y1:y2, x1:x2] = matrix_val
            
        self.last_img_x = img_x
        self.last_img_y = img_y
        
        self.update_canvas(matrix_changed=True)
        self.update_cursor(event)

    def save_matrix(self):
        save_path = filedialog.asksaveasfilename(
            defaultextension=".txt",
            initialfile=self.file_path,
            filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
        )
        if save_path:
            np.savetxt(save_path, self.matrix, fmt="%d")
            print(f"Matrix successfully saved to {save_path}")

if __name__ == "__main__":
    root = tk.Tk()
    root.geometry("900x700")
    
    target_txt_file = r"../file_io/line_img_matrix.txt"
    
    app = MatrixTextEditor(root, target_txt_file)
    root.mainloop()

In [4]:
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageTk

class MatrixTextEditor:
    def __init__(self, root, file_path):
        self.root = root
        self.root.title("Matrix Text File Editor (Undo/Redo & Hidden Cursor)")
        self.file_path = file_path
        
        # 1. Load the matrix from the text file using NumPy
        try:
            self.matrix = np.loadtxt(self.file_path, dtype=int)
        except Exception as e:
            print(f"Error loading matrix: {e}")
            self.matrix = np.zeros((100, 100), dtype=int)

        self.height, self.width = self.matrix.shape
        
        # Normalize matrix values for display if they are 0 and 1
        self.val_max = self.matrix.max()
        if self.val_max <= 1:
            self.display_scale = 255
        else:
            self.display_scale = 1
            
        self.brush_size = 3
        self.draw_value = 0  # Default to drawing black (erasing artifacts)
        
        # Zoom & Pan State variables
        self.zoom_scale = 1.0
        self.pan_x = 0.0
        self.pan_y = 0.0
        self.drag_start_x = 0
        self.drag_start_y = 0
        
        # Stroke tracking for interpolation
        self.last_img_x = None
        self.last_img_y = None
        
        # Cursor preview item reference
        self.cursor_rect = None
        
        # Caching for performance
        self._cached_zoom = None
        self._cached_tk_img = None

        # Undo / Redo History Stacks
        self.undo_stack = []
        self.redo_stack = []
        self.max_history = 30
        
        self.setup_ui()
        self.update_canvas()

    def setup_ui(self):
        # Control Panel
        control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
        control_frame.pack(side=tk.TOP, fill=tk.X)
        
        tk.Button(control_frame, text="Eraser / Black (0)", command=lambda: self.set_draw_value(0), bg="#ffcccc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        tk.Button(control_frame, text="Draw / White (1/Max)", command=lambda: self.set_draw_value(1), bg="#ccffcc", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=5)
        
        tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(side=tk.LEFT, padx=10)
        self.size_slider = tk.Scale(control_frame, from_=1, to=20, orient=tk.HORIZONTAL, command=self.update_brush)
        self.size_slider.set(self.brush_size)
        self.size_slider.pack(side=tk.LEFT, padx=5)
        
        tk.Button(control_frame, text="Reset View", command=self.reset_view, bg="#e0e0e0", font=("Arial", 10, "bold")).pack(side=tk.LEFT, padx=10)
        tk.Button(control_frame, text="Save Matrix to .txt", command=self.save_matrix, bg="#cce5ff", font=("Arial", 10, "bold")).pack(side=tk.RIGHT, padx=5)

        # Canvas Area for the Image
        canvas_frame = tk.Frame(self.root)
        canvas_frame.pack(fill=tk.BOTH, expand=True)
        
        self.canvas = tk.Canvas(canvas_frame, bg="gray", highlightthickness=0)
        self.canvas.pack(fill=tk.BOTH, expand=True)
        
        # Bind Mouse Actions for Painting (Left Click & Drag)
        self.canvas.bind("<Button-1>", self.start_stroke)      
        self.canvas.bind("<B1-Motion>", self.paint)
        self.canvas.bind("<ButtonRelease-1>", self.end_stroke)

        # Bind Mouse Actions for Panning (Scrollwheel / Middle Click / Right Click)
        self.canvas.bind("<Button-2>", self.start_pan)
        self.canvas.bind("<B2-Motion>", self.do_pan)
        self.canvas.bind("<Button-3>", self.start_pan)
        self.canvas.bind("<B3-Motion>", self.do_pan)

        # Bind Mouse Wheel for Zooming
        self.canvas.bind("<MouseWheel>", self.zoom)     # Windows and macOS
        self.canvas.bind("<Button-4>", self.zoom)       # Linux scroll up
        self.canvas.bind("<Button-5>", self.zoom)       # Linux scroll down

        # Bind Cursor Hover Preview Events & Hide OS Mouse
        self.canvas.bind("<Motion>", self.update_cursor)
        self.canvas.bind("<Enter>", self.show_cursor)
        self.canvas.bind("<Leave>", self.hide_cursor)

        # Bind Keyboard Shortcuts for Undo (Ctrl+Z / Cmd+Z) and Redo (Ctrl+Y / Ctrl+Shift+Z)
        self.root.bind("<Control-z>", self.undo)
        self.root.bind("<Command-z>", self.undo)  # macOS support
        self.root.bind("<Control-y>", self.redo)
        self.root.bind("<Control-Shift-Z>", self.redo)

    def set_draw_value(self, val):
        if val == 1 and self.val_max > 1:
            self.draw_value = self.val_max
        else:
            self.draw_value = val

    def update_brush(self, val):
        self.brush_size = int(val)

    def reset_view(self):
        self.zoom_scale = 1.0
        self.pan_x = 0.0
        self.pan_y = 0.0
        self.update_canvas()

    def push_undo_state(self):
        # Save a copy of the matrix before changes for undo history
        self.undo_stack.append(self.matrix.copy())
        if len(self.undo_stack) > self.max_history:
            self.undo_stack.pop(0)
        # Clear redo stack on new user action
        self.redo_stack.clear()

    def undo(self, event=None):
        if not self.undo_stack:
            return
        # Push current state to redo stack
        self.redo_stack.append(self.matrix.copy())
        # Pop previous state from undo stack
        self.matrix = self.undo_stack.pop()
        self.update_canvas(matrix_changed=True)

    def redo(self, event=None):
        if not self.redo_stack:
            return
        # Push current state to undo stack
        self.undo_stack.append(self.matrix.copy())
        # Pop next state from redo stack
        self.matrix = self.redo_stack.pop()
        self.update_canvas(matrix_changed=True)

    def update_canvas(self, matrix_changed=False):
        scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
        pil_img = Image.fromarray(scaled_matrix)
        
        new_width = max(1, int(self.width * self.zoom_scale))
        new_height = max(1, int(self.height * self.zoom_scale))
        
        if matrix_changed or self._cached_zoom != self.zoom_scale or self._cached_tk_img is None:
            resized_pil = pil_img.resize((new_width, new_height), Image.NEAREST)
            self._cached_tk_img = ImageTk.PhotoImage(resized_pil)
            self._cached_zoom = self.zoom_scale

        self.canvas.delete("all")
        self.image_item = self.canvas.create_image(self.pan_x, self.pan_y, anchor=tk.NW, image=self._cached_tk_img)

        # Recreate cursor preview rect if active
        if self.cursor_rect is not None:
            self.cursor_rect = self.canvas.create_rectangle(0, 0, 0, 0, outline="red", width=1)

    def show_cursor(self, event):
        # Hide standard OS mouse cursor inside canvas
        self.canvas.config(cursor="none")
        if self.cursor_rect is None:
            self.cursor_rect = self.canvas.create_rectangle(0, 0, 0, 0, outline="red", width=1)
        self.update_cursor(event)

    def hide_cursor(self, event):
        # Restore standard OS mouse cursor when leaving canvas area
        self.canvas.config(cursor="")
        if self.cursor_rect is not None:
            self.canvas.delete(self.cursor_rect)
            self.cursor_rect = None

    def update_cursor(self, event):
        if self.cursor_rect is None:
            self.show_cursor(event)
            
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        
        x, y = int(img_x), int(img_y)
        r = self.brush_size
        
        x1 = (x - r) * self.zoom_scale + self.pan_x
        y1 = (y - r) * self.zoom_scale + self.pan_y
        x2 = (x + r) * self.zoom_scale + self.pan_x
        y2 = (y + r) * self.zoom_scale + self.pan_y
        
        self.canvas.coords(self.cursor_rect, x1, y1, x2, y2)

    def start_pan(self, event):
        self.drag_start_x = event.x
        self.drag_start_y = event.y

    def do_pan(self, event):
        dx = event.x - self.drag_start_x
        dy = event.y - self.drag_start_y
        self.pan_x += dx
        self.pan_y += dy
        self.drag_start_x = event.x
        self.drag_start_y = event.y
        
        self.canvas.coords(self.image_item, self.pan_x, self.pan_y)
        self.update_cursor(event)

    def zoom(self, event):
        if event.num == 4 or event.delta > 0:
            zoom_factor = 1.15
        elif event.num == 5 or event.delta < 0:
            zoom_factor = 1 / 1.15
        else:
            zoom_factor = 1.0

        old_zoom = self.zoom_scale
        new_zoom = self.zoom_scale * zoom_factor
        
        if new_zoom < 0.1 or new_zoom > 50.0:
            return

        self.zoom_scale = new_zoom
        self.pan_x = event.x - (event.x - self.pan_x) * (self.zoom_scale / old_zoom)
        self.pan_y = event.y - (event.y - self.pan_y) * (self.zoom_scale / old_zoom)

        self.update_canvas(matrix_changed=False)
        self.update_cursor(event)

    def start_stroke(self, event):
        # Save state to undo history before starting stroke modifications
        self.push_undo_state()
        
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        self.last_img_x = img_x
        self.last_img_y = img_y
        self.paint(event)

    def paint(self, event):
        img_x = (event.x - self.pan_x) / self.zoom_scale
        img_y = (event.y - self.pan_y) / self.zoom_scale
        
        if self.last_img_x is None or self.last_img_y is None:
            self.last_img_x = img_x
            self.last_img_y = img_y

        dist = np.hypot(img_x - self.last_img_x, img_y - self.last_img_y)
        steps = max(1, int(dist))
        
        xs = np.linspace(self.last_img_x, img_x, steps)
        ys = np.linspace(self.last_img_y, img_y, steps)
        
        matrix_val = self.draw_value if self.display_scale == 1 else (1 if self.draw_value > 0 else 0)
        r = self.brush_size
        
        for cx, cy in zip(xs, ys):
            x, y = int(cx), int(cy)
            y1, y2 = max(0, y - r), min(self.height, y + r)
            x1, x2 = max(0, x - r), min(self.width, x + r)
            self.matrix[y1:y2, x1:x2] = matrix_val
            
        self.last_img_x = img_x
        self.last_img_y = img_y
        
        self.update_canvas(matrix_changed=True)
        self.update_cursor(event)

    def end_stroke(self, event):
        self.last_img_x = None
        self.last_img_y = None

    def save_matrix(self):
        save_path = filedialog.asksaveasfilename(
            defaultextension=".txt",
            initialfile=self.file_path,
            filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")]
        )
        if save_path:
            np.savetxt(save_path, self.matrix, fmt="%d")
            print(f"Matrix successfully saved to {save_path}")

if __name__ == "__main__":
    root = tk.Tk()
    root.geometry("900x700")
    
    target_txt_file = r"../file_io/line_img_matrix.txt"
    
    app = MatrixTextEditor(root, target_txt_file)
    root.mainloop()

In [2]:
from collections import deque
import math
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageTk
import pdf2image


class MatrixTextEditor:

  def __init__(self, root, file_path):
    self.root = root
    self.root.title("Matrix Text File Editor")
    self.file_path = file_path

    # 1. Load the matrix from the text file using NumPy
    try:
      self.matrix = np.loadtxt(self.file_path, dtype=int)
    except Exception as e:
      print(f"Error loading matrix: {e}")
      self.matrix = np.zeros((100, 100), dtype=int)

    self.height, self.width = self.matrix.shape

    # Normalize matrix values for display if they are 0 and 1
    self.val_max = self.matrix.max()
    if self.val_max <= 1:
      self.display_scale = 255
    else:
      self.display_scale = 1

    self.brush_size = 3
    self.draw_value = 0  # Default to drawing black (erasing artifacts)

    # Zoom & Pan State variables
    self.zoom_scale = 1.0
    self.pan_x = 0.0
    self.pan_y = 0.0
    self.drag_start_x = 0
    self.drag_start_y = 0

    # Stroke tracking for interpolation
    self.last_img_x = None
    self.last_img_y = None

    # Cursor preview item reference
    self.cursor_rect = None

    # Caching for performance
    self._cached_zoom = None
    self._cached_tk_img = None

    # Undo / Redo History Stacks
    self.undo_stack = []
    self.redo_stack = []
    self.max_history = 30

    self.setup_ui()
    self.update_canvas()

  def setup_ui(self):
    # Control Panel
    control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
    control_frame.pack(side=tk.TOP, fill=tk.X)

    tk.Button(
        control_frame,
        text="Eraser / Black (0)",
        command=lambda: self.set_draw_value(0),
        bg="#ffcccc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)
    tk.Button(
        control_frame,
        text="Draw / White (1/Max)",
        command=lambda: self.set_draw_value(1),
        bg="#ccffcc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(
        side=tk.LEFT, padx=10
    )
    self.size_slider = tk.Scale(
        control_frame,
        from_=1,
        to=20,
        orient=tk.HORIZONTAL,
        command=self.update_brush,
    )
    self.size_slider.set(self.brush_size)
    self.size_slider.pack(side=tk.LEFT, padx=5)

    tk.Button(
        control_frame,
        text="Reset View",
        command=self.reset_view,
        bg="#e0e0e0",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    # Added button for Method 2 Ray Casting Leak Detection
    tk.Button(
        control_frame,
        text="Detect Leaks (Ray Cast)",
        command=self.run_ray_cast_detection,
        bg="#ffebcc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    tk.Button(
        control_frame,
        text="Save Matrix to .txt",
        command=self.save_matrix,
        bg="#cce5ff",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.RIGHT, padx=5)

    # Canvas Area for the Image
    canvas_frame = tk.Frame(self.root)
    canvas_frame.pack(fill=tk.BOTH, expand=True)

    self.canvas = tk.Canvas(canvas_frame, bg="gray", highlightthickness=0)
    self.canvas.pack(fill=tk.BOTH, expand=True)

    # Bind Mouse Actions for Painting (Left Click & Drag)
    self.canvas.bind("<Button-1>", self.start_stroke)
    self.canvas.bind("<B1-Motion>", self.paint)
    self.canvas.bind("<ButtonRelease-1>", self.end_stroke)

    # Bind Mouse Actions for Panning (Scrollwheel / Middle Click / Right Click)
    self.canvas.bind("<Button-2>", self.start_pan)
    self.canvas.bind("<B2-Motion>", self.do_pan)
    self.canvas.bind("<Button-3>", self.start_pan)
    self.canvas.bind("<B3-Motion>", self.do_pan)

    # Bind Mouse Wheel for Zooming
    self.canvas.bind("<MouseWheel>", self.zoom)  # Windows and macOS
    self.canvas.bind("<Button-4>", self.zoom)  # Linux scroll up
    self.canvas.bind("<Button-5>", self.zoom)  # Linux scroll down

    # Bind Cursor Hover Preview Events & Hide OS Mouse
    self.canvas.bind("<Motion>", self.update_cursor)
    self.canvas.bind("<Enter>", self.show_cursor)
    self.canvas.bind("<Leave>", self.hide_cursor)

    # Bind Keyboard Shortcuts for Undo (Ctrl+Z / Cmd+Z) and Redo (Ctrl+Y / Ctrl+Shift+Z)
    self.root.bind("<Control-z>", self.undo)
    self.root.bind("<Command-z>", self.undo)  # macOS support
    self.root.bind("<Control-y>", self.redo)
    self.root.bind("<Control-Shift-Z>", self.redo)

  def set_draw_value(self, val):
    if val == 1 and self.val_max > 1:
      self.draw_value = self.val_max
    else:
      self.draw_value = val

  def update_brush(self, val):
    self.brush_size = int(val)

  def reset_view(self):
    self.zoom_scale = 1.0
    self.pan_x = 0.0
    self.pan_y = 0.0
    self.update_canvas()

  def push_undo_state(self):
    # Save a copy of the matrix before changes for undo history
    self.undo_stack.append(self.matrix.copy())
    if len(self.undo_stack) > self.max_history:
      self.undo_stack.pop(0)
    # Clear redo stack on new user action
    self.redo_stack.clear()

  def undo(self, event=None):
    if not self.undo_stack:
      return
    # Push current state to redo stack
    self.redo_stack.append(self.matrix.copy())
    # Pop previous state from undo stack
    self.matrix = self.undo_stack.pop()
    self.update_canvas(matrix_changed=True)

  def redo(self, event=None):
    if not self.redo_stack:
      return
    # Push current state to undo stack
    self.undo_stack.append(self.matrix.copy())
    # Pop next state from redo stack
    self.matrix = self.redo_stack.pop()
    self.update_canvas(matrix_changed=True)

  def update_canvas(self, matrix_changed=False):
    scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
    pil_img = Image.fromarray(scaled_matrix)

    new_width = max(1, int(self.width * self.zoom_scale))
    new_height = max(1, int(self.height * self.zoom_scale))

    if (
        matrix_changed
        or self._cached_zoom != self.zoom_scale
        or self._cached_tk_img is None
    ):
      resized_pil = pil_img.resize((new_width, new_height), Image.NEAREST)
      self._cached_tk_img = ImageTk.PhotoImage(resized_pil)
      self._cached_zoom = self.zoom_scale

    self.canvas.delete("all")
    self.image_item = self.canvas.create_image(
        self.pan_x, self.pan_y, anchor=tk.NW, image=self._cached_tk_img
    )

    # Recreate cursor preview rect if active
    if self.cursor_rect is not None:
      self.cursor_rect = self.canvas.create_rectangle(
          0, 0, 0, 0, outline="red", width=1
      )

  def show_cursor(self, event):
    # Hide standard OS mouse cursor inside canvas
    self.canvas.config(cursor="none")
    if self.cursor_rect is None:
      self.cursor_rect = self.canvas.create_rectangle(
          0, 0, 0, 0, outline="red", width=1
      )
    self.update_cursor(event)

  def hide_cursor(self, event):
    # Restore standard OS mouse cursor when leaving canvas area
    self.canvas.config(cursor="")
    if self.cursor_rect is not None:
      self.canvas.delete(self.cursor_rect)
      self.cursor_rect = None

  def update_cursor(self, event):
    if self.cursor_rect is None:
      self.show_cursor(event)

    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    x, y = int(img_x), int(img_y)
    r = self.brush_size

    x1 = (x - r) * self.zoom_scale + self.pan_x
    y1 = (y - r) * self.zoom_scale + self.pan_y
    x2 = (x + r) * self.zoom_scale + self.pan_x
    y2 = (y + r) * self.zoom_scale + self.pan_y

    self.canvas.coords(self.cursor_rect, x1, y1, x2, y2)

  def start_pan(self, event):
    self.drag_start_x = event.x
    self.drag_start_y = event.y

  def do_pan(self, event):
    dx = event.x - self.drag_start_x
    dy = event.y - self.drag_start_y
    self.pan_x += dx
    self.pan_y += dy
    self.drag_start_x = event.x
    self.drag_start_y = event.y

    self.canvas.coords(self.image_item, self.pan_x, self.pan_y)
    self.update_cursor(event)

  def zoom(self, event):
    if event.num == 4 or event.delta > 0:
      zoom_factor = 1.15
    elif event.num == 5 or event.delta < 0:
      zoom_factor = 1 / 1.15
    else:
      zoom_factor = 1.0

    old_zoom = self.zoom_scale
    new_zoom = self.zoom_scale * zoom_factor

    if new_zoom < 0.1 or new_zoom > 50.0:
      return

    self.zoom_scale = new_zoom
    self.pan_x = event.x - (event.x - self.pan_x) * (
        self.zoom_scale / old_zoom
    )
    self.pan_y = event.y - (event.y - self.pan_y) * (
        self.zoom_scale / old_zoom
    )

    self.update_canvas(matrix_changed=False)
    self.update_cursor(event)

  def start_stroke(self, event):
    # Save state to undo history before starting stroke modifications
    self.push_undo_state()

    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale
    self.last_img_x = img_x
    self.last_img_y = img_y
    self.paint(event)

  def paint(self, event):
    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    if self.last_img_x is None or self.last_img_y is None:
      self.last_img_x = img_x
      self.last_img_y = img_y

    dist = np.hypot(img_x - self.last_img_x, img_y - self.last_img_y)
    steps = max(1, int(dist))

    xs = np.linspace(self.last_img_x, img_x, steps)
    ys = np.linspace(self.last_img_y, img_y, steps)

    matrix_val = (
        self.draw_value
        if self.display_scale == 1
        else (1 if self.draw_value > 0 else 0)
    )
    r = self.brush_size

    for cx, cy in zip(xs, ys):
      x, y = int(cx), int(cy)
      y1, y2 = max(0, y - r), min(self.height, y + r)
      x1, x2 = max(0, x - r), min(self.width, x + r)
      self.matrix[y1:y2, x1:x2] = matrix_val

    self.last_img_x = img_x
    self.last_img_y = img_y

    self.update_canvas(matrix_changed=True)
    self.update_cursor(event)

  def end_stroke(self, event):
    self.last_img_x = None
    self.last_img_y = None

  def find_routes_and_holes(self, center_point, num_rays=360):
    """Casts rays outward from a room center (y, x) to find wall breaches."""
    height, width = self.matrix.shape
    cy, cx = center_point
    open_routes = []

    max_distance = int(math.hypot(height, width))

    for angle in range(num_rays):
      rad = math.radians(angle)
      for r in range(1, max_distance):
        ny = int(cy + r * math.sin(rad))
        nx = int(cx + r * math.cos(rad))

        # Check matrix bounds (escaped to boundary)
        if not (0 <= ny < height and 0 <= nx < width):
          open_routes.append({"angle": angle, "coord": (ny, nx)})
          break

        # Check if ray hits a wall (1)
        if self.matrix[ny, nx] == 1:
          break

    return open_routes

  def run_ray_cast_detection(self):
    """Executes ray casting from an automatic room center and flags holes."""
    center_y, center_x = self.height // 2, self.width // 2

    # Find open space near the center if center falls on a wall
    if self.matrix[center_y, center_x] == 1:
      indices = np.argwhere(self.matrix == 0)
      if len(indices) > 0:
        distances = np.hypot(indices[:, 0] - center_y, indices[:, 1] - center_x)
        center_y, center_x = indices[np.argmin(distances)]
      else:
        print("No open space found in matrix!")
        return

    routes = self.find_routes_and_holes((center_y, center_x), num_rays=360)
    print(
        f"Ray-casting complete from seed ({center_y}, {center_x}). Found"
        f" {len(routes)} leaked directions out of 360."
    )

    # Optional: Draw visual markers on the canvas where rays escaped
    for route in routes:
      # Clamp visual points or display the exit boundary points
      ly, lx = route["coord"]
      ly = max(0, min(self.height - 1, ly))
      lx = max(0, min(self.width - 1, lx))

      screen_x = lx * self.zoom_scale + self.pan_x
      screen_y = ly * self.zoom_scale + self.pan_y
      r = 2
      self.canvas.create_oval(
          screen_x - r,
          screen_y - r,
          screen_x + r,
          screen_y + r,
          outline="yellow",
          fill="yellow",
      )

  def save_matrix(self):
    save_path = filedialog.asksaveasfilename(
        defaultextension=".txt",
        initialfile=self.file_path,
        filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")],
    )
    if save_path:
      np.savetxt(save_path, self.matrix, fmt="%d")
      print(f"Matrix successfully saved to {save_path}")


if __name__ == "__main__":
  root = tk.Tk()
  root.geometry("900x700")

  target_txt_file = r"../file_io/line_img_matrix.txt"

  app = MatrixTextEditor(root, target_txt_file)
  root.mainloop()

Ray-casting complete from seed (272, 424). Found 0 leaked directions out of 360.
Ray-casting complete from seed (272, 424). Found 0 leaked directions out of 360.
Ray-casting complete from seed (272, 424). Found 0 leaked directions out of 360.


In [5]:
from collections import deque
import math
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageTk
import pdf2image


class MatrixTextEditor:

  def __init__(self, root, file_path):
    self.root = root
    self.root.title("Matrix Text File Editor")
    self.file_path = file_path

    # 1. Load the matrix from the text file using NumPy
    try:
      self.matrix = np.loadtxt(self.file_path, dtype=int)
    except Exception as e:
      print(f"Error loading matrix: {e}")
      self.matrix = np.zeros((100, 100), dtype=int)

    self.height, self.width = self.matrix.shape

    # Normalize matrix values for display if they are 0 and 1
    self.val_max = self.matrix.max()
    if self.val_max <= 1:
      self.display_scale = 255
    else:
      self.display_scale = 1

    self.brush_size = 3
    self.draw_value = 0  # Default to drawing black (erasing artifacts)

    # Zoom & Pan State variables
    self.zoom_scale = 1.0
    self.pan_x = 0.0
    self.pan_y = 0.0
    self.drag_start_x = 0
    self.drag_start_y = 0

    # Stroke tracking for interpolation
    self.last_img_x = None
    self.last_img_y = None

    # Cursor preview item reference
    self.cursor_rect = None

    # Caching for performance
    self._cached_zoom = None
    self._cached_tk_img = None

    # Undo / Redo History Stacks
    self.undo_stack = []
    self.redo_stack = []
    self.max_history = 30

    self.setup_ui()
    self.update_canvas()

  def setup_ui(self):
    # Control Panel
    control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
    control_frame.pack(side=tk.TOP, fill=tk.X)

    tk.Button(
        control_frame,
        text="Eraser / Black (0)",
        command=lambda: self.set_draw_value(0),
        bg="#ffcccc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)
    tk.Button(
        control_frame,
        text="Draw / White (1/Max)",
        command=lambda: self.set_draw_value(1),
        bg="#ccffcc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(
        side=tk.LEFT, padx=10
    )
    self.size_slider = tk.Scale(
        control_frame,
        from_=1,
        to=20,
        orient=tk.HORIZONTAL,
        command=self.update_brush,
    )
    self.size_slider.set(self.brush_size)
    self.size_slider.pack(side=tk.LEFT, padx=5)

    tk.Button(
        control_frame,
        text="Reset View",
        command=self.reset_view,
        bg="#e0e0e0",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    tk.Button(
        control_frame,
        text="Save Matrix to .txt",
        command=self.save_matrix,
        bg="#cce5ff",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.RIGHT, padx=5)

    # Canvas Area for the Image
    canvas_frame = tk.Frame(self.root)
    canvas_frame.pack(fill=tk.BOTH, expand=True)

    self.canvas = tk.Canvas(canvas_frame, bg="gray", highlightthickness=0)
    self.canvas.pack(fill=tk.BOTH, expand=True)

    # Bind Mouse Actions for Painting (Left Click & Drag)
    self.canvas.bind("<Button-1>", self.start_stroke)
    self.canvas.bind("<B1-Motion>", self.paint)
    self.canvas.bind("<ButtonRelease-1>", self.end_stroke)

    # Bind Right-Click to Ray-Cast Leak Inspection from the clicked room
    self.canvas.bind("<Button-3>", self.run_ray_cast_at_click)

    # Bind Middle-Click for Panning
    self.canvas.bind("<Button-2>", self.start_pan)
    self.canvas.bind("<B2-Motion>", self.do_pan)

    # Bind Mouse Wheel for Zooming
    self.canvas.bind("<MouseWheel>", self.zoom)  # Windows and macOS
    self.canvas.bind("<Button-4>", self.zoom)  # Linux scroll up
    self.canvas.bind("<Button-5>", self.zoom)  # Linux scroll down

    # Bind Cursor Hover Preview Events & Hide OS Mouse
    self.canvas.bind("<Motion>", self.update_cursor)
    self.canvas.bind("<Enter>", self.show_cursor)
    self.canvas.bind("<Leave>", self.hide_cursor)

    # Bind Keyboard Shortcuts for Undo (Ctrl+Z / Cmd+Z) and Redo (Ctrl+Y / Ctrl+Shift+Z)
    self.root.bind("<Control-z>", self.undo)
    self.root.bind("<Command-z>", self.undo)  # macOS support
    self.root.bind("<Control-y>", self.redo)
    self.root.bind("<Control-Shift-Z>", self.redo)

  def set_draw_value(self, val):
    if val == 1 and self.val_max > 1:
      self.draw_value = self.val_max
    else:
      self.draw_value = val

  def update_brush(self, val):
    self.brush_size = int(val)

  def reset_view(self):
    self.zoom_scale = 1.0
    self.pan_x = 0.0
    self.pan_y = 0.0
    self.update_canvas()

  def push_undo_state(self):
    self.undo_stack.append(self.matrix.copy())
    if len(self.undo_stack) > self.max_history:
      self.undo_stack.pop(0)
    self.redo_stack.clear()

  def undo(self, event=None):
    if not self.undo_stack:
      return
    self.redo_stack.append(self.matrix.copy())
    self.matrix = self.undo_stack.pop()
    self.update_canvas(matrix_changed=True)

  def redo(self, event=None):
    if not self.redo_stack:
      return
    self.undo_stack.append(self.matrix.copy())
    self.matrix = self.redo_stack.pop()
    self.update_canvas(matrix_changed=True)

  def update_canvas(self, matrix_changed=False):
    scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
    pil_img = Image.fromarray(scaled_matrix)

    new_width = max(1, int(self.width * self.zoom_scale))
    new_height = max(1, int(self.height * self.zoom_scale))

    if (
        matrix_changed
        or self._cached_zoom != self.zoom_scale
        or self._cached_tk_img is None
    ):
      resized_pil = pil_img.resize((new_width, new_height), Image.NEAREST)
      self._cached_tk_img = ImageTk.PhotoImage(resized_pil)
      self._cached_zoom = self.zoom_scale

    self.canvas.delete("all")
    self.image_item = self.canvas.create_image(
        self.pan_x, self.pan_y, anchor=tk.NW, image=self._cached_tk_img
    )

    if self.cursor_rect is not None:
      self.cursor_rect = self.canvas.create_rectangle(
          0, 0, 0, 0, outline="red", width=1
      )

  def show_cursor(self, event):
    self.canvas.config(cursor="none")
    if self.cursor_rect is None:
      self.cursor_rect = self.canvas.create_rectangle(
          0, 0, 0, 0, outline="red", width=1
      )
    self.update_cursor(event)

  def hide_cursor(self, event):
    self.canvas.config(cursor="")
    if self.cursor_rect is not None:
      self.canvas.delete(self.cursor_rect)
      self.cursor_rect = None

  def update_cursor(self, event):
    if self.cursor_rect is None:
      self.show_cursor(event)

    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    x, y = int(img_x), int(img_y)
    r = self.brush_size

    x1 = (x - r) * self.zoom_scale + self.pan_x
    y1 = (y - r) * self.zoom_scale + self.pan_y
    x2 = (x + r) * self.zoom_scale + self.pan_x
    y2 = (y + r) * self.zoom_scale + self.pan_y

    self.canvas.coords(self.cursor_rect, x1, y1, x2, y2)

  def start_pan(self, event):
    self.drag_start_x = event.x
    self.drag_start_y = event.y

  def do_pan(self, event):
    dx = event.x - self.drag_start_x
    dy = event.y - self.drag_start_y
    self.pan_x += dx
    self.pan_y += dy
    self.drag_start_x = event.x
    self.drag_start_y = event.y

    self.canvas.coords(self.image_item, self.pan_x, self.pan_y)
    self.update_cursor(event)

  def zoom(self, event):
    if event.num == 4 or event.delta > 0:
      zoom_factor = 1.15
    elif event.num == 5 or event.delta < 0:
      zoom_factor = 1 / 1.15
    else:
      zoom_factor = 1.0

    old_zoom = self.zoom_scale
    new_zoom = self.zoom_scale * zoom_factor

    if new_zoom < 0.1 or new_zoom > 50.0:
      return

    self.zoom_scale = new_zoom
    self.pan_x = event.x - (event.x - self.pan_x) * (
        self.zoom_scale / old_zoom
    )
    self.pan_y = event.y - (event.y - self.pan_y) * (
        self.zoom_scale / old_zoom
    )

    self.update_canvas(matrix_changed=False)
    self.update_cursor(event)

  def start_stroke(self, event):
    self.push_undo_state()

    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale
    self.last_img_x = img_x
    self.last_img_y = img_y
    self.paint(event)

  def paint(self, event):
    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    if self.last_img_x is None or self.last_img_y is None:
      self.last_img_x = img_x
      self.last_img_y = img_y

    dist = np.hypot(img_x - self.last_img_x, img_y - self.last_img_y)
    steps = max(1, int(dist))

    xs = np.linspace(self.last_img_x, img_x, steps)
    ys = np.linspace(self.last_img_y, img_y, steps)

    matrix_val = (
        self.draw_value
        if self.display_scale == 1
        else (1 if self.draw_value > 0 else 0)
    )
    r = self.brush_size

    for cx, cy in zip(xs, ys):
      x, y = int(cx), int(cy)
      y1, y2 = max(0, y - r), min(self.height, y + r)
      x1, x2 = max(0, x - r), min(self.width, x + r)
      self.matrix[y1:y2, x1:x2] = matrix_val

    self.last_img_x = img_x
    self.last_img_y = img_y

    self.update_canvas(matrix_changed=True)
    self.update_cursor(event)

  def end_stroke(self, event):
    self.last_img_x = None
    self.last_img_y = None

  def find_routes_and_holes(self, center_point, num_rays=360):
    """Casts rays outward from a room center (y, x) to find wall breaches."""
    height, width = self.matrix.shape
    cy, cx = center_point
    open_routes = []

    max_distance = int(math.hypot(height, width))

    for angle in range(num_rays):
      rad = math.radians(angle)
      for r in range(1, max_distance):
        ny = int(cy + r * math.sin(rad))
        nx = int(cx + r * math.cos(rad))

        if not (0 <= ny < height and 0 <= nx < width):
          open_routes.append({"angle": angle, "coord": (ny, nx)})
          break

        if self.matrix[ny, nx] == 1:
          break

    return open_routes

  def run_ray_cast_at_click(self, event):
    """Triggered by Right-Click: Casts rays from the exact cursor coordinate."""
    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    x, y = int(img_x), int(img_y)

    if not (0 <= x < self.width and 0 <= y < self.height):
      print("Clicked outside matrix boundaries!")
      return

    if self.matrix[y, x] == 1:
      print(f"Clicked coordinate ({y}, {x}) is inside a wall! Click on open space (room).")
      return

    # Refresh canvas to remove old markers
    self.update_canvas(matrix_changed=True)

    # Run ray-casting from the clicked coordinate (y, x)
    routes = self.find_routes_and_holes((y, x), num_rays=360)
    print(
        f"Ray-casting complete from custom seed ({y}, {x}). Found"
        f" {len(routes)} leaked directions out of 360."
    )

    # Draw markers for escaped boundary/hole points (Yellow)
    for route in routes:
      ly, lx = route["coord"]
      ly = max(0, min(self.height - 1, ly))
      lx = max(0, min(self.width - 1, lx))

      screen_x = lx * self.zoom_scale + self.pan_x
      screen_y = ly * self.zoom_scale + self.pan_y
      r = 2
      self.canvas.create_oval(
          screen_x - r,
          screen_y - r,
          screen_x + r,
          screen_y + r,
          outline="yellow",
          fill="yellow",
      )

    # Mark the clicked room seed point (Cyan)
    seed_screen_x = x * self.zoom_scale + self.pan_x
    seed_screen_y = y * self.zoom_scale + self.pan_y
    self.canvas.create_oval(
        seed_screen_x - 4,
        seed_screen_y - 4,
        seed_screen_x + 4,
        seed_screen_y + 4,
        outline="cyan",
        fill="cyan",
    )

  def save_matrix(self):
    save_path = filedialog.asksaveasfilename(
        defaultextension=".txt",
        initialfile=self.file_path,
        filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")],
    )
    if save_path:
      np.savetxt(save_path, self.matrix, fmt="%d")
      print(f"Matrix successfully saved to {save_path}")


if __name__ == "__main__":
  root = tk.Tk()
  root.geometry("900x700")

  target_txt_file = r"../file_io/line_img_matrix.txt"

  app = MatrixTextEditor(root, target_txt_file)
  root.mainloop()

Ray-casting complete from custom seed (233, 670). Found 2 leaked directions out of 360.
Ray-casting complete from custom seed (205, 666). Found 1 leaked directions out of 360.


In [2]:
#raycast lines

from collections import deque
import math
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageTk
import pdf2image


class MatrixTextEditor:

  def __init__(self, root, file_path):
    self.root = root
    self.root.title("Matrix Text File Editor")
    self.file_path = file_path

    # 1. Load the matrix from the text file using NumPy
    try:
      self.matrix = np.loadtxt(self.file_path, dtype=int)
    except Exception as e:
      print(f"Error loading matrix: {e}")
      self.matrix = np.zeros((100, 100), dtype=int)

    self.height, self.width = self.matrix.shape

    # Normalize matrix values for display if they are 0 and 1
    self.val_max = self.matrix.max()
    if self.val_max <= 1:
      self.display_scale = 255
    else:
      self.display_scale = 1

    self.brush_size = 3
    self.draw_value = 0  # Default to drawing black (erasing artifacts)

    # Zoom & Pan State variables
    self.zoom_scale = 1.0
    self.pan_x = 0.0
    self.pan_y = 0.0
    self.drag_start_x = 0
    self.drag_start_y = 0

    # Stroke tracking for interpolation
    self.last_img_x = None
    self.last_img_y = None

    # Cursor preview item reference
    self.cursor_rect = None

    # Caching for performance
    self._cached_zoom = None
    self._cached_tk_img = None

    # Undo / Redo History Stacks
    self.undo_stack = []
    self.redo_stack = []
    self.max_history = 30

    self.setup_ui()
    self.update_canvas()

  def setup_ui(self):
    # Control Panel
    control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
    control_frame.pack(side=tk.TOP, fill=tk.X)

    tk.Button(
        control_frame,
        text="Eraser / Black (0)",
        command=lambda: self.set_draw_value(0),
        bg="#ffcccc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)
    tk.Button(
        control_frame,
        text="Draw / White (1/Max)",
        command=lambda: self.set_draw_value(1),
        bg="#ccffcc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(
        side=tk.LEFT, padx=10
    )
    self.size_slider = tk.Scale(
        control_frame,
        from_=1,
        to=20,
        orient=tk.HORIZONTAL,
        command=self.update_brush,
    )
    self.size_slider.set(self.brush_size)
    self.size_slider.pack(side=tk.LEFT, padx=5)

    tk.Button(
        control_frame,
        text="Reset View",
        command=self.reset_view,
        bg="#e0e0e0",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    tk.Button(
        control_frame,
        text="Save Matrix to .txt",
        command=self.save_matrix,
        bg="#cce5ff",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.RIGHT, padx=5)

    # Canvas Area for the Image
    canvas_frame = tk.Frame(self.root)
    canvas_frame.pack(fill=tk.BOTH, expand=True)

    self.canvas = tk.Canvas(canvas_frame, bg="gray", highlightthickness=0)
    self.canvas.pack(fill=tk.BOTH, expand=True)

    # Bind Mouse Actions for Painting (Left Click & Drag)
    self.canvas.bind("<Button-1>", self.start_stroke)
    self.canvas.bind("<B1-Motion>", self.paint)
    self.canvas.bind("<ButtonRelease-1>", self.end_stroke)

    # Bind Right-Click to Ray-Cast Line Inspection from the clicked room
    self.canvas.bind("<Button-3>", self.run_ray_cast_at_click)

    # Bind Middle-Click for Panning
    self.canvas.bind("<Button-2>", self.start_pan)
    self.canvas.bind("<B2-Motion>", self.do_pan)

    # Bind Mouse Wheel for Zooming
    self.canvas.bind("<MouseWheel>", self.zoom)  # Windows and macOS
    self.canvas.bind("<Button-4>", self.zoom)  # Linux scroll up
    self.canvas.bind("<Button-5>", self.zoom)  # Linux scroll down

    # Bind Cursor Hover Preview Events & Hide OS Mouse
    self.canvas.bind("<Motion>", self.update_cursor)
    self.canvas.bind("<Enter>", self.show_cursor)
    self.canvas.bind("<Leave>", self.hide_cursor)

    # Bind Keyboard Shortcuts for Undo (Ctrl+Z / Cmd+Z) and Redo (Ctrl+Y / Ctrl+Shift+Z)
    self.root.bind("<Control-z>", self.undo)
    self.root.bind("<Command-z>", self.undo)  # macOS support
    self.root.bind("<Control-y>", self.redo)
    self.root.bind("<Control-Shift-Z>", self.redo)

  def set_draw_value(self, val):
    if val == 1 and self.val_max > 1:
      self.draw_value = self.val_max
    else:
      self.draw_value = val

  def update_brush(self, val):
    self.brush_size = int(val)

  def reset_view(self):
    self.zoom_scale = 1.0
    self.pan_x = 0.0
    self.pan_y = 0.0
    self.update_canvas()

  def push_undo_state(self):
    self.undo_stack.append(self.matrix.copy())
    if len(self.undo_stack) > self.max_history:
      self.undo_stack.pop(0)
    self.redo_stack.clear()

  def undo(self, event=None):
    if not self.undo_stack:
      return
    self.redo_stack.append(self.matrix.copy())
    self.matrix = self.undo_stack.pop()
    self.update_canvas(matrix_changed=True)

  def redo(self, event=None):
    if not self.redo_stack:
      return
    self.undo_stack.append(self.matrix.copy())
    self.matrix = self.redo_stack.pop()
    self.update_canvas(matrix_changed=True)

  def update_canvas(self, matrix_changed=False):
    scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
    pil_img = Image.fromarray(scaled_matrix)

    new_width = max(1, int(self.width * self.zoom_scale))
    new_height = max(1, int(self.height * self.zoom_scale))

    if (
        matrix_changed
        or self._cached_zoom != self.zoom_scale
        or self._cached_tk_img is None
    ):
      resized_pil = pil_img.resize((new_width, new_height), Image.NEAREST)
      self._cached_tk_img = ImageTk.PhotoImage(resized_pil)
      self._cached_zoom = self.zoom_scale

    self.canvas.delete("all")
    self.image_item = self.canvas.create_image(
        self.pan_x, self.pan_y, anchor=tk.NW, image=self._cached_tk_img
    )

    if self.cursor_rect is not None:
      self.cursor_rect = self.canvas.create_rectangle(
          0, 0, 0, 0, outline="red", width=1
      )

  def show_cursor(self, event):
    self.canvas.config(cursor="none")
    if self.cursor_rect is None:
      self.cursor_rect = self.canvas.create_rectangle(
          0, 0, 0, 0, outline="red", width=1
      )
    self.update_cursor(event)

  def hide_cursor(self, event):
    self.canvas.config(cursor="")
    if self.cursor_rect is not None:
      self.canvas.delete(self.cursor_rect)
      self.cursor_rect = None

  def update_cursor(self, event):
    if self.cursor_rect is None:
      self.show_cursor(event)

    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    x, y = int(img_x), int(img_y)
    r = self.brush_size

    x1 = (x - r) * self.zoom_scale + self.pan_x
    y1 = (y - r) * self.zoom_scale + self.pan_y
    x2 = (x + r) * self.zoom_scale + self.pan_x
    y2 = (y + r) * self.zoom_scale + self.pan_y

    self.canvas.coords(self.cursor_rect, x1, y1, x2, y2)

  def start_pan(self, event):
    self.drag_start_x = event.x
    self.drag_start_y = event.y

  def do_pan(self, event):
    dx = event.x - self.drag_start_x
    dy = event.y - self.drag_start_y
    self.pan_x += dx
    self.pan_y += dy
    self.drag_start_x = event.x
    self.drag_start_y = event.y

    self.canvas.coords(self.image_item, self.pan_x, self.pan_y)
    self.update_cursor(event)

  def zoom(self, event):
    if event.num == 4 or event.delta > 0:
      zoom_factor = 1.15
    elif event.num == 5 or event.delta < 0:
      zoom_factor = 1 / 1.15
    else:
      zoom_factor = 1.0

    old_zoom = self.zoom_scale
    new_zoom = self.zoom_scale * zoom_factor

    if new_zoom < 0.1 or new_zoom > 50.0:
      return

    self.zoom_scale = new_zoom
    self.pan_x = event.x - (event.x - self.pan_x) * (
        self.zoom_scale / old_zoom
    )
    self.pan_y = event.y - (event.y - self.pan_y) * (
        self.zoom_scale / old_zoom
    )

    self.update_canvas(matrix_changed=False)
    self.update_cursor(event)

  def start_stroke(self, event):
    self.push_undo_state()

    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale
    self.last_img_x = img_x
    self.last_img_y = img_y
    self.paint(event)

  def paint(self, event):
    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    if self.last_img_x is None or self.last_img_y is None:
      self.last_img_x = img_x
      self.last_img_y = img_y

    dist = np.hypot(img_x - self.last_img_x, img_y - self.last_img_y)
    steps = max(1, int(dist))

    xs = np.linspace(self.last_img_x, img_x, steps)
    ys = np.linspace(self.last_img_y, img_y, steps)

    matrix_val = (
        self.draw_value
        if self.display_scale == 1
        else (1 if self.draw_value > 0 else 0)
    )
    r = self.brush_size

    for cx, cy in zip(xs, ys):
      x, y = int(cx), int(cy)
      y1, y2 = max(0, y - r), min(self.height, y + r)
      x1, x2 = max(0, x - r), min(self.width, x + r)
      self.matrix[y1:y2, x1:x2] = matrix_val

    self.last_img_x = img_x
    self.last_img_y = img_y

    self.update_canvas(matrix_changed=True)
    self.update_cursor(event)

  def end_stroke(self, event):
    self.last_img_x = None
    self.last_img_y = None

  def find_routes_and_holes(self, center_point, num_rays=360):
    """Casts rays outward from a room center (y, x).

    Tracks where each ray terminates (wall hit or boundary escape).
    """
    height, width = self.matrix.shape
    cy, cx = center_point
    ray_results = []

    max_distance = int(math.hypot(height, width))

    for angle in range(num_rays):
      rad = math.radians(angle)
      end_y, end_x = cy, cx
      escaped = True

      for r in range(1, max_distance):
        ny = int(cy + r * math.sin(rad))
        nx = int(cx + r * math.cos(rad))

        # Check bounds (escaped to outer boundary)
        if not (0 <= ny < height and 0 <= nx < width):
          end_y, end_x = ny, nx
          escaped = True
          break

        # Check if ray hits a wall
        if self.matrix[ny, nx] == 1:
          end_y, end_x = ny, nx
          escaped = False
          break

      ray_results.append({
          "angle": angle,
          "end": (end_y, end_x),
          "escaped": escaped,
      })

    return ray_results

  def run_ray_cast_at_click(self, event):
    """Triggered by Right-Click: Casts rays from cursor and draws lines."""
    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    x, y = int(img_x), int(img_y)

    if not (0 <= x < self.width and 0 <= y < self.height):
      print("Clicked outside matrix boundaries!")
      return

    if self.matrix[y, x] == 1:
      print(
          f"Clicked coordinate ({y}, {x}) is inside a wall! Click on open"
          " space (room)."
      )
      return

    # Refresh canvas to clear previous drawings
    self.update_canvas(matrix_changed=True)

    # Run ray-casting from the clicked coordinate (y, x)
    rays = self.find_routes_and_holes((y, x), num_rays=540)
    print(
        f"Ray-casting complete from custom seed ({y}, {x}). Cast {len(rays)}"
        " rays."
    )

    seed_screen_x = x * self.zoom_scale + self.pan_x
    seed_screen_y = y * self.zoom_scale + self.pan_y

    # Draw lines for every ray from the seed point to its termination point
    for ray in rays:
      ey, ex = ray["end"]
      ey = max(0, min(self.height - 1, ey))
      ex = max(0, min(self.width - 1, ex))

      end_screen_x = ex * self.zoom_scale + self.pan_x
      end_screen_y = ey * self.zoom_scale + self.pan_y

      # Color leaks bright yellow, normal wall bounds dim cyan
      if ray["escaped"]:
        line_color = "yellow"
        line_width = 1.5
      else:
        line_color = "#00cccc"
        line_width = 1

      self.canvas.create_line(
          seed_screen_x,
          seed_screen_y,
          end_screen_x,
          end_screen_y,
          fill=line_color,
          width=line_width,
      )

    # Mark the clicked room seed point with a red dot
    self.canvas.create_oval(
        seed_screen_x - 4,
        seed_screen_y - 4,
        seed_screen_x + 4,
        seed_screen_y + 4,
        outline="red",
        fill="red",
    )

  def save_matrix(self):
    save_path = filedialog.asksaveasfilename(
        defaultextension=".txt",
        initialfile=self.file_path,
        filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")],
    )
    if save_path:
      np.savetxt(save_path, self.matrix, fmt="%d")
      print(f"Matrix successfully saved to {save_path}")


if __name__ == "__main__":
  root = tk.Tk()
  root.geometry("900x700")

  target_txt_file = r"../file_io/line_img_matrix.txt"

  app = MatrixTextEditor(root, target_txt_file)
  root.mainloop()

Ray-casting complete from custom seed (184, 674). Cast 540 rays.


In [3]:
# cross

from collections import deque
import math
import tkinter as tk
from tkinter import filedialog
import numpy as np
from PIL import Image, ImageTk
import pdf2image


class MatrixTextEditor:

  def __init__(self, root, file_path):
    self.root = root
    self.root.title("Matrix Text File Editor")
    self.file_path = file_path

    # 1. Load the matrix from the text file using NumPy
    try:
      self.matrix = np.loadtxt(self.file_path, dtype=int)
    except Exception as e:
      print(f"Error loading matrix: {e}")
      self.matrix = np.zeros((100, 100), dtype=int)

    self.height, self.width = self.matrix.shape

    # Normalize matrix values for display if they are 0 and 1
    self.val_max = self.matrix.max()
    if self.val_max <= 1:
      self.display_scale = 255
    else:
      self.display_scale = 1

    self.brush_size = 3
    self.draw_value = 0  # Default to drawing black (erasing artifacts)

    # Zoom & Pan State variables
    self.zoom_scale = 1.0
    self.pan_x = 0.0
    self.pan_y = 0.0
    self.drag_start_x = 0
    self.drag_start_y = 0

    # Stroke tracking for interpolation
    self.last_img_x = None
    self.last_img_y = None

    # Cursor preview item reference
    self.cursor_rect = None

    # Caching for performance
    self._cached_zoom = None
    self._cached_tk_img = None

    # Undo / Redo History Stacks
    self.undo_stack = []
    self.redo_stack = []
    self.max_history = 30

    self.setup_ui()
    self.update_canvas()

  def setup_ui(self):
    # Control Panel
    control_frame = tk.Frame(self.root, bg="#f0f0f0", pady=5)
    control_frame.pack(side=tk.TOP, fill=tk.X)

    tk.Button(
        control_frame,
        text="Eraser / Black (0)",
        command=lambda: self.set_draw_value(0),
        bg="#ffcccc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)
    tk.Button(
        control_frame,
        text="Draw / White (1/Max)",
        command=lambda: self.set_draw_value(1),
        bg="#ccffcc",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    tk.Label(control_frame, text="Brush Size:", bg="#f0f0f0").pack(
        side=tk.LEFT, padx=10
    )
    self.size_slider = tk.Scale(
        control_frame,
        from_=1,
        to=20,
        orient=tk.HORIZONTAL,
        command=self.update_brush,
    )
    self.size_slider.set(self.brush_size)
    self.size_slider.pack(side=tk.LEFT, padx=5)

    tk.Button(
        control_frame,
        text="Reset View",
        command=self.reset_view,
        bg="#e0e0e0",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.LEFT, padx=5)

    tk.Button(
        control_frame,
        text="Save Matrix to .txt",
        command=self.save_matrix,
        bg="#cce5ff",
        font=("Arial", 10, "bold"),
    ).pack(side=tk.RIGHT, padx=5)

    # Canvas Area for the Image
    canvas_frame = tk.Frame(self.root)
    canvas_frame.pack(fill=tk.BOTH, expand=True)

    self.canvas = tk.Canvas(canvas_frame, bg="gray", highlightthickness=0)
    self.canvas.pack(fill=tk.BOTH, expand=True)

    # Bind Mouse Actions for Painting (Left Click & Drag)
    self.canvas.bind("<Button-1>", self.start_stroke)
    self.canvas.bind("<B1-Motion>", self.paint)
    self.canvas.bind("<ButtonRelease-1>", self.end_stroke)

    # Bind Right-Click to Ray-Cast Line Inspection & Red Cardinal Cross
    self.canvas.bind("<Button-3>", self.run_ray_cast_at_click)

    # Bind Middle-Click for Panning
    self.canvas.bind("<Button-2>", self.start_pan)
    self.canvas.bind("<B2-Motion>", self.do_pan)

    # Bind Mouse Wheel for Zooming
    self.canvas.bind("<MouseWheel>", self.zoom)  # Windows and macOS
    self.canvas.bind("<Button-4>", self.zoom)  # Linux scroll up
    self.canvas.bind("<Button-5>", self.zoom)  # Linux scroll down

    # Bind Cursor Hover Preview Events & Hide OS Mouse
    self.canvas.bind("<Motion>", self.update_cursor)
    self.canvas.bind("<Enter>", self.show_cursor)
    self.canvas.bind("<Leave>", self.hide_cursor)

    # Bind Keyboard Shortcuts for Undo (Ctrl+Z / Cmd+Z) and Redo (Ctrl+Y / Ctrl+Shift+Z)
    self.root.bind("<Control-z>", self.undo)
    self.root.bind("<Command-z>", self.undo)  # macOS support
    self.root.bind("<Control-y>", self.redo)
    self.root.bind("<Control-Shift-Z>", self.redo)

  def set_draw_value(self, val):
    if val == 1 and self.val_max > 1:
      self.draw_value = self.val_max
    else:
      self.draw_value = val

  def update_brush(self, val):
    self.brush_size = int(val)

  def reset_view(self):
    self.zoom_scale = 1.0
    self.pan_x = 0.0
    self.pan_y = 0.0
    self.update_canvas()

  def push_undo_state(self):
    self.undo_stack.append(self.matrix.copy())
    if len(self.undo_stack) > self.max_history:
      self.undo_stack.pop(0)
    self.redo_stack.clear()

  def undo(self, event=None):
    if not self.undo_stack:
      return
    self.redo_stack.append(self.matrix.copy())
    self.matrix = self.undo_stack.pop()
    self.update_canvas(matrix_changed=True)

  def redo(self, event=None):
    if not self.redo_stack:
      return
    self.undo_stack.append(self.matrix.copy())
    self.matrix = self.redo_stack.pop()
    self.update_canvas(matrix_changed=True)

  def update_canvas(self, matrix_changed=False):
    scaled_matrix = (self.matrix * self.display_scale).astype(np.uint8)
    pil_img = Image.fromarray(scaled_matrix)

    new_width = max(1, int(self.width * self.zoom_scale))
    new_height = max(1, int(self.height * self.zoom_scale))

    if (
        matrix_changed
        or self._cached_zoom != self.zoom_scale
        or self._cached_tk_img is None
    ):
      resized_pil = pil_img.resize((new_width, new_height), Image.NEAREST)
      self._cached_tk_img = ImageTk.PhotoImage(resized_pil)
      self._cached_zoom = self.zoom_scale

    self.canvas.delete("all")
    self.image_item = self.canvas.create_image(
        self.pan_x, self.pan_y, anchor=tk.NW, image=self._cached_tk_img
    )

    if self.cursor_rect is not None:
      self.cursor_rect = self.canvas.create_rectangle(
          0, 0, 0, 0, outline="red", width=1
      )

  def show_cursor(self, event):
    self.canvas.config(cursor="none")
    if self.cursor_rect is None:
      self.cursor_rect = self.canvas.create_rectangle(
          0, 0, 0, 0, outline="red", width=1
      )
    self.update_cursor(event)

  def hide_cursor(self, event):
    self.canvas.config(cursor="")
    if self.cursor_rect is not None:
      self.canvas.delete(self.cursor_rect)
      self.cursor_rect = None

  def update_cursor(self, event):
    if self.cursor_rect is None:
      self.show_cursor(event)

    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    x, y = int(img_x), int(img_y)
    r = self.brush_size

    x1 = (x - r) * self.zoom_scale + self.pan_x
    y1 = (y - r) * self.zoom_scale + self.pan_y
    x2 = (x + r) * self.zoom_scale + self.pan_x
    y2 = (y + r) * self.zoom_scale + self.pan_y

    self.canvas.coords(self.cursor_rect, x1, y1, x2, y2)

  def start_pan(self, event):
    self.drag_start_x = event.x
    self.drag_start_y = event.y

  def do_pan(self, event):
    dx = event.x - self.drag_start_x
    dy = event.y - self.drag_start_y
    self.pan_x += dx
    self.pan_y += dy
    self.drag_start_x = event.x
    self.drag_start_y = event.y

    self.canvas.coords(self.image_item, self.pan_x, self.pan_y)
    self.update_cursor(event)

  def zoom(self, event):
    if event.num == 4 or event.delta > 0:
      zoom_factor = 1.15
    elif event.num == 5 or event.delta < 0:
      zoom_factor = 1 / 1.15
    else:
      zoom_factor = 1.0

    old_zoom = self.zoom_scale
    new_zoom = self.zoom_scale * zoom_factor

    if new_zoom < 0.1 or new_zoom > 50.0:
      return

    self.zoom_scale = new_zoom
    self.pan_x = event.x - (event.x - self.pan_x) * (
        self.zoom_scale / old_zoom
    )
    self.pan_y = event.y - (event.y - self.pan_y) * (
        self.zoom_scale / old_zoom
    )

    self.update_canvas(matrix_changed=False)
    self.update_cursor(event)

  def start_stroke(self, event):
    self.push_undo_state()

    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale
    self.last_img_x = img_x
    self.last_img_y = img_y
    self.paint(event)

  def paint(self, event):
    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    if self.last_img_x is None or self.last_img_y is None:
      self.last_img_x = img_x
      self.last_img_y = img_y

    dist = np.hypot(img_x - self.last_img_x, img_y - self.last_img_y)
    steps = max(1, int(dist))

    xs = np.linspace(self.last_img_x, img_x, steps)
    ys = np.linspace(self.last_img_y, img_y, steps)

    matrix_val = (
        self.draw_value
        if self.display_scale == 1
        else (1 if self.draw_value > 0 else 0)
    )
    r = self.brush_size

    for cx, cy in zip(xs, ys):
      x, y = int(cx), int(cy)
      y1, y2 = max(0, y - r), min(self.height, y + r)
      x1, x2 = max(0, x - r), min(self.width, x + r)
      self.matrix[y1:y2, x1:x2] = matrix_val

    self.last_img_x = img_x
    self.last_img_y = img_y

    self.update_canvas(matrix_changed=True)
    self.update_cursor(event)

  def end_stroke(self, event):
    self.last_img_x = None
    self.last_img_y = None

  def find_routes_and_holes(self, seed_point, num_rays=360):
    """Casts rays outward from a given seed point (y, x)."""
    height, width = self.matrix.shape
    cy, cx = seed_point
    ray_results = []

    max_distance = int(math.hypot(height, width))

    for angle in range(num_rays):
      rad = math.radians(angle * (360.0 / num_rays))
      end_y, end_x = cy, cx
      escaped = True

      for r in range(1, max_distance):
        ny = int(cy + r * math.sin(rad))
        nx = int(cx + r * math.cos(rad))

        if not (0 <= ny < height and 0 <= nx < width):
          end_y, end_x = ny, nx
          escaped = True
          break

        if self.matrix[ny, nx] == 1:
          end_y, end_x = ny, nx
          escaped = False
          break

      ray_results.append({
          "start": (cy, cx),
          "end": (end_y, end_x),
          "escaped": escaped,
      })

    return ray_results

  def cast_cardinal_rays(self, center_point):
    """Casts rays strictly in 4 cardinal directions and returns the endpoints

    along with all intermediate points along the cross lines.
    """
    height, width = self.matrix.shape
    cy, cx = center_point
    cross_endpoints = []
    cross_points = [(cy, cx)]
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right
    max_distance = max(height, width)

    for dy, dx in directions:
      end_y, end_x = cy, cx
      for r in range(1, max_distance):
        ny = cy + r * dy
        nx = cx + r * dx
        cross_points.append((ny, nx))
        if not (0 <= ny < height and 0 <= nx < width):
          end_y, end_x = ny, nx
          break
        if self.matrix[ny, nx] == 1:
          end_y, end_x = ny, nx
          break
      cross_endpoints.append((end_y, end_x))

    return cross_endpoints, cross_points

  def run_ray_cast_at_click(self, event):
    """Triggered by Right-Click: Casts rays from the center AND every point along the cardinal cross."""
    img_x = (event.x - self.pan_x) / self.zoom_scale
    img_y = (event.y - self.pan_y) / self.zoom_scale

    x, y = int(img_x), int(img_y)

    if not (0 <= x < self.width and 0 <= y < self.height):
      print("Clicked outside matrix boundaries!")
      return

    if self.matrix[y, x] == 1:
      print(
          f"Clicked coordinate ({y}, {x}) is inside a wall! Click on open"
          " space (room)."
      )
      return

    # Refresh canvas to clear previous drawings
    self.update_canvas(matrix_changed=True)

    # 1. Get the cardinal cross endpoints and all points along the cross lines
    cardinal_ends, cross_points = self.cast_cardinal_rays((y, x))

    # 2. To keep performance smooth, sample points along the cross (e.g., every 3rd pixel or all of them)
    # Using every point or a sampled subset ensures all cross points act as raycast origins.
    sampled_seeds = cross_points[::2]  # Adjust step if needed for speed

    # 3. Run ray casting from each seed point on the cross
    all_rays = []
    for seed in sampled_seeds:
      # Use fewer rays (e.g. 180) for cross secondary seeds to maintain high performance
      rays = self.find_routes_and_holes(seed, num_rays=180)
      all_rays.extend(rays)

    # Draw all ray lines on canvas
    for ray in all_rays:
      scy, scx = ray["start"]
      ey, ex = ray["end"]

      start_screen_x = scx * self.zoom_scale + self.pan_x
      start_screen_y = scy * self.zoom_scale + self.pan_y
      end_screen_x = ex * self.zoom_scale + self.pan_x
      end_screen_y = ey * self.zoom_scale + self.pan_y

      if ray["escaped"]:
        line_color = "yellow"
        line_width = 1.0
      else:
        line_color = "#00cccc"
        line_width = 0.75

      self.canvas.create_line(
          start_screen_x,
          start_screen_y,
          end_screen_x,
          end_screen_y,
          fill=line_color,
          width=line_width,
      )

    # 4. Draw the prominent Red Cardinal Cross lines on top
    seed_screen_x = x * self.zoom_scale + self.pan_x
    seed_screen_y = y * self.zoom_scale + self.pan_y

    for ey, ex in cardinal_ends:
      ey = max(0, min(self.height - 1, ey))
      ex = max(0, min(self.width - 1, ex))

      end_screen_x = ex * self.zoom_scale + self.pan_x
      end_screen_y = ey * self.zoom_scale + self.pan_y

      self.canvas.create_line(
          seed_screen_x,
          seed_screen_y,
          end_screen_x,
          end_screen_y,
          fill="red",
          width=2.5,
      )

    # 5. Mark the clicked room seed point with a red dot on top
    self.canvas.create_oval(
        seed_screen_x - 4,
        seed_screen_y - 4,
        seed_screen_x + 4,
        seed_screen_y + 4,
        outline="red",
        fill="red",
    )

  def save_matrix(self):
    save_path = filedialog.asksaveasfilename(
        defaultextension=".txt",
        initialfile=self.file_path,
        filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")],
    )
    if save_path:
      np.savetxt(save_path, self.matrix, fmt="%d")
      print(f"Matrix successfully saved to {save_path}")


if __name__ == "__main__":
  root = tk.Tk()
  root.geometry("900x700")

  target_txt_file = r"../file_io/line_img_matrix.txt"

  app = MatrixTextEditor(root, target_txt_file)
  root.mainloop()